<a href="https://colab.research.google.com/github/MamoMGD1/ISE302-DataMining-GroupProject/blob/main/students/models/model_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Model 1 — Price Estimation (XGBoost Regression)

**Objective:** Estimate a car's market price in Turkish Lira (TL) based on its key features.

## Introduction
In this notebook, we are building our regression model using XGBoost. I specifically chose to use the unscaled dataset here because tree-based models like XGBoost do not require feature scaling, which allows us to keep the original values for easier interpretation later. Our target variable is `log_Fiyat`. By training on the log-transformed price, we prevent extreme outliers (like ultra-luxury cars) from skewing our model. We will reverse this transformation at the evaluation stage to get real TL values.

# 📝 Project Updates & Optimizations (Changelog)
*Note for the team: The following optimizations were made to the original XGBoost template to ensure code execution stability, prevent overfitting, and maximize our R² score.*

**1. Categorical Data Fix (Bug Prevention):**
*   **Change:** Added a loop to explicitly cast `object` (string) columns to `category` datatypes before the train/test split.
*   **Reason:** XGBoost with `enable_categorical=True` crashes if categorical variables are passed as standard strings. This ensures seamless execution.

**2. Feature Selection (TODO 1):**
*   **Change:** Kept the recommended base features but added `Marka` (Brand) to the feature list.
*   **Reason:** Brand perception is one of the strongest drivers of car prices in the Turkish market. Leaving it out leaves significant predictive power on the table.

**3. Hyperparameter Tuning (TODO 2):**
*   **Change:** Tuned the model away from the default placeholders to a more balanced configuration for our 2,589-row dataset:
    *   `n_estimators`: Increased from 100 to 400.
    *   `learning_rate`: Decreased from 0.1 to 0.05.
    *   `max_depth`: Decreased from 6 to 5.
    *   `colsample_bytree`: Added and set to 0.8.
*   **Reason:** A lower learning rate with more estimators allows the model to learn complex relationships more carefully. Dropping the depth to 5 and adding `colsample_bytree` (which randomly samples features per tree) heavily protects our model from overfitting on our relatively small dataset.

## 1. Data Import
Here, we load our unscaled dataset and the necessary libraries. I am verifying the shape of the dataset to ensure all 2,589 rows and 87 columns are loaded correctly before moving forward.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

df = pd.read_csv('https://raw.githubusercontent.com/MamoMGD1/ISE302-DataMining-GroupProject/main/data/proceed_dataset_without_scaling.csv')
print(f"Dataset shape: {df.shape}")
df.head()

## 2. Feature Selection & Data Preparation
I carefully selected the features that logically drive a car's market value. Based on my review of the data, I made sure to include the `Marka` (Brand) column, as brand prestige heavily dictates car prices in Türkiye.

**Important Fix:** I also added a preprocessing loop to explicitly convert all text/object columns into pandas `category` types. Without this step, XGBoost's `enable_categorical=True` parameter will throw an error. Finally, I split the data using an 80/20 train-test split.

In [ ]:
# Selected features including 'Marka' to capture brand value
recommended_features = [
    'Marka', 'Yıl', 'Kilometre', 'Motor Hacmi', 'Motor Gücü', 'Tork',
    'Hızlanma (0-100)', 'Seri', 'Model', 'Vites Tipi',
    'Kasa Tipi_SUV', 'Çekiş_AWD (Elektronik)',
    'Yakıt Tipi_Dizel', 'Yakıt Tipi_Hibrit',
    'Ortalama Kasko', 'paint_damage_score', 'total_changed_parts'
]

# Filter to columns that actually exist in the dataset
features = [f for f in recommended_features if f in df.columns]
target = 'log_Fiyat'

X = df[features].copy()
y = df[target].copy()

# CRITICAL FIX: Convert object (string) columns to category type for XGBoost
for col in X.select_dtypes(['object']).columns:
    X[col] = X[col].astype('category')

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training set: {X_train.shape}, Test set: {X_test.shape}")

## 3. Model Training (XGBoost)
Instead of using the default baseline parameters, I applied a custom hyperparameter tuning strategy to prevent our model from overfitting on our relatively small dataset of 2,500 rows.

I increased `n_estimators` to 400 but lowered the `learning_rate` to 0.05, which forces the model to learn gradual, more robust patterns. I also restricted `max_depth` to 5 and added `colsample_bytree=0.8`. This ensures the trees don't just rely heavily on obvious features (like the car's year) but also learn from secondary features like damage scores.

In [ ]:
from xgboost import XGBRegressor

# Tuned Hyperparameters to balance performance and prevent overfitting
model = XGBRegressor(
    n_estimators=400,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,   # Adds feature randomness to prevent relying on single features
    random_state=42,
    enable_categorical=True
)

# Train the model
model.fit(X_train, y_train)

# Generate log-scale predictions
y_pred_log = model.predict(X_test)
print("Model training complete.")

## 4. Evaluation — Converting Predictions back to TL
Since we trained our model on `log_Fiyat`, the raw predictions are in log format. To understand our model's real-world accuracy, I applied `np.expm1()` to reverse the log transformation for both our predictions and the actual test values. We can now evaluate our RMSE, MAE, and R² scores in actual Turkish Lira.

In [ ]:
# Convert log-scale test values and predictions back to actual Turkish Lira
y_test_tl = np.expm1(y_test)
y_pred_tl = np.expm1(y_pred_log)

# Calculate metrics on the actual TL values
rmse_tl = np.sqrt(mean_squared_error(y_test_tl, y_pred_tl))
mae_tl = mean_absolute_error(y_test_tl, y_pred_tl)
r2 = r2_score(y_test_tl, y_pred_tl)

print(f"RMSE: {rmse_tl:,.0f} TL")
print(f"MAE:  {mae_tl:,.0f} TL")
print(f"R²:   {r2:.4f}")

## 5. Visualization — Actual vs. Predicted Scatter Plot
This scatter plot visualizes our model's overall accuracy. The dashed blue line represents a perfect prediction. The closer our points hug this line, the better our model is performing. I added a color gradient to help us quickly spot which price ranges have the highest absolute prediction errors.

In [ ]:
residuals_tl = np.abs(y_test_tl - y_pred_tl)

fig, ax = plt.subplots(figsize=(10, 7))
scatter = ax.scatter(y_test_tl, y_pred_tl, c=residuals_tl, cmap='RdYlGn_r',
                     alpha=0.6, s=40, edgecolors='none')

plt.colorbar(scatter, ax=ax, label='Absolute Error (TL)')
max_val = max(y_test_tl.max(), y_pred_tl.max())
ax.plot([0, max_val], [0, max_val], 'b--', linewidth=2, label='Perfect Prediction')

ax.set_xlabel('Actual Price (TL)', fontsize=13)
ax.set_ylabel('Predicted Price (TL)', fontsize=13)
ax.set_title('Actual vs Predicted Car Prices (XGBoost)', fontsize=15)
ax.legend()
plt.tight_layout()
plt.show()

## 6. Visualization — Residual Distribution Histogram
This histogram checks the distribution of our model's errors (Residuals). Ideally, we want to see a bell curve perfectly centered around the red "Zero Error" line. If the curve is heavily skewed to the left or right, it would indicate that our model systematically overprices or underprices cars.

In [ ]:
errors_tl = y_pred_tl - y_test_tl

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(errors_tl, bins=50, color='steelblue', edgecolor='white')
ax.axvline(0, color='red', linestyle='--', linewidth=2, label='Zero Error')

ax.set_xlabel('Prediction Error (TL)', fontsize=13)
ax.set_ylabel('Count', fontsize=13)
ax.set_title('Residual Distribution (Predicted − Actual)', fontsize=15)
ax.legend()
plt.tight_layout()
plt.show()

## 7. Feature Importance Analysis
This bar chart extracts the internal logic of our XGBoost model. It shows us exactly which features the model prioritized when calculating the price. This is a great sanity check to validate that the model's behavior aligns with real-world automotive market dynamics.

In [ ]:
importances = model.feature_importances_
feat_imp = pd.Series(importances, index=features).sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 6))
feat_imp.plot(kind='barh', ax=ax, color='teal')
ax.invert_yaxis()

ax.set_xlabel('XGBoost Importance Score', fontsize=13)
ax.set_title('Top 15 Feature Importances (XGBoost)', fontsize=15)
plt.tight_layout()
plt.show()

## 8. Sample Prediction Visualization (Actual vs. Predicted)
Instead of looking at a raw table of numbers, which can be hard to read, I created a grouped bar chart to compare the actual vs. predicted prices for a random sample of 10 test cars. The blue bars represent the actual market price, while the red bars show our model's prediction. This makes it instantly obvious where our model overestimates or underestimates.

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt

# Grab a random sample of 10 cars from the test set
random.seed(42)
idx = random.sample(range(len(X_test)), 10)

actual_prices = y_test_tl.iloc[idx].values
predicted_prices = y_pred_tl[idx]

# Create labels for the x-axis
car_labels = [f"Car {i+1}" for i in range(10)]

x = np.arange(len(car_labels))  # the label locations
width = 0.35  # the width of the bars

# Set up the plot
fig, ax = plt.subplots(figsize=(12, 6))

# Create the grouped bars (Blue for Actual, Red for Predicted)
rects1 = ax.bar(x - width/2, actual_prices, width, label='Actual Price (Blue)', color='dodgerblue', edgecolor='black')
rects2 = ax.bar(x + width/2, predicted_prices, width, label='Predicted Price (Red)', color='tomato', edgecolor='black')

# Add text, labels, and titles
ax.set_ylabel('Price (TL)', fontsize=13)
ax.set_title('Sample of 10 Cars: Actual vs Predicted Prices', fontsize=16, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(car_labels, fontsize=11)
ax.legend(fontsize=12)

# Format the y-axis to show regular numbers with commas instead of scientific notation (e.g. 1e6)
ax.get_yaxis().set_major_formatter(plt.FuncFormatter(lambda x, loc: "{:,}".format(int(x))))

# Add a faint grid for easier reading
ax.yaxis.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

---
## 🌟 BONUS 1: Advanced Feature Impact (SHAP Analysis)
While our standard Feature Importance chart tells us *which* variables are important, it doesn't tell us *how* they affect the price. To understand the direction of the impact, I am implementing SHAP (SHapley Additive exPlanations), an industry-standard interpretability framework.

This summary plot will show us exactly how high or low values of a specific feature push the car's predicted price up or down.

In [ ]:
# Install shap if not already installed in Colab
!pip install shap -q

import shap

# Initialize the SHAP explainer with our trained XGBoost model
explainer = shap.TreeExplainer(model)

# Calculate SHAP values for the test set (using a sample if the dataset is huge, but 500 rows is fine)
shap_values = explainer.shap_values(X_test)

# Plot the SHAP summary
plt.figure(figsize=(10, 6))
plt.title("SHAP Summary Plot: How Features Impact Car Price", fontsize=15, pad=20)
shap.summary_plot(shap_values, X_test, show=False)
plt.tight_layout()
plt.show()

## 🌟 BONUS 2: Interactive Price Predictor
To make our model usable for a non-technical audience, I have built a lightweight, interactive user interface directly inside this notebook using `ipywidgets`.

You can use the sliders and dropdowns below to input a hypothetical car's details, and the model will instantly generate a live price prediction in Turkish Lira.
*(Note: I have locked some of the background categorical variables to common defaults to keep the UI clean).*

In [ ]:
import ipywidgets as widgets
from IPython.display import display
import pandas as pd
import numpy as np

def predict_price_live(Yil, Kilometre, Motor_Hacmi, Motor_Gucu, Tork, Kasko_Degeri):
    # 1. Start with the values from our interactive sliders
    input_data = {
        'Yıl': Yil,
        'Kilometre': Kilometre,
        'Motor Hacmi': Motor_Hacmi,
        'Motor Gücü': Motor_Gucu,
        'Tork': Tork,
        'Ortalama Kasko': Kasko_Degeri,
    }

    # 2. Dynamically fill in the remaining background features the model needs
    for col in features:
        if col not in input_data:
            # If the feature is categorical, default to the most common value in the dataset
            if X[col].dtype.name == 'category':
                input_data[col] = X[col].mode()[0]
            # If the feature is numerical, default to the median value
            else:
                input_data[col] = X[col].median()

    # 3. Convert to a DataFrame in the exact column order the model expects
    input_df = pd.DataFrame([input_data])[features]

    # 4. Cast columns to category type just like we did in training
    for col in input_df.select_dtypes(['object', 'category']).columns:
        input_df[col] = input_df[col].astype('category')

    # 5. Predict and reverse the log transformation
    log_pred = model.predict(input_df)
    tl_pred = np.expm1(log_pred)[0]

    print("-" * 40)
    print(f"💰 LIVE PREDICTED PRICE: {tl_pred:,.0f} TL 💰")
    print("-" * 40)

# Create the interactive UI
widgets.interact(
    predict_price_live,
    Yil=widgets.IntSlider(min=2000, max=2025, step=1, value=2018, description='Year:'),
    Kilometre=widgets.IntSlider(min=0, max=300000, step=5000, value=80000, description='Mileage:'),
    Motor_Hacmi=widgets.IntSlider(min=900, max=4000, step=100, value=1600, description='Engine (cc):'),
    Motor_Gucu=widgets.IntSlider(min=70, max=400, step=10, value=120, description='Power (HP):'),
    Tork=widgets.IntSlider(min=100, max=600, step=10, value=250, description='Torque:'),
    Kasko_Degeri=widgets.FloatSlider(min=0.1, max=2.0, step=0.1, value=0.5, description='Kasko Ratio:')
);

## 🌟 BONUS 3: Model Export (MLOps)
In a real-world scenario, we do not want to retrain the model every time the server restarts. Below, I am using the `joblib` library to serialize and export our trained XGBoost model as a `.pkl` file.

This file can now be handed off to a software engineering team to be integrated into a web application or mobile app backend.

In [ ]:
import joblib

# Define the filename
filename = 'xgboost_price_model.pkl'

# Save the model to disk
joblib.dump(model, filename)

print(f"✅ Success! The model has been saved as '{filename}'.")
print("You can download this file from the Colab file browser on the left menu.")